# 68 - Held-out PRO160 Q-Planning evaluation, worker 0/4

This is shard 0 of 4 over the reserved **160 position-perturbation LIBERO-PRO identities**. These six suites were excluded as a category from corrector training. Every identity runs three matched arms: ordinary stock PI0.5, the frozen Q10 planner, and the frozen Q50 planner.

Stock uses 10 Euler steps. Both planners sample 64 candidates with 3 Euler steps, retain the best 16 by Q, and execute their Q-softmax weighted chunk. All arms execute 10 actions and then replan. There is no online learning, PnP refinement, uncertainty gate, or historical-row reuse. Videos and observation frames are off; compact trajectories and planner diagnostics are retained.

The worker contains 40 identities and 120 rollouts. It prints an exact three-arm matched SR table every 10 fully completed identities. For a smoke test set EPISODE_LIMIT = 1, then restore None; rerunning safely skips completed rows.

## 1. Setup

In [ ]:
EXTRAS = 'sim'
SETUP_ENV = True
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

## 2. Checkpoints and fixed shard

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

SHARD_COUNT = 4
SHARD_INDEX = 0
EPISODE_LIMIT = None  # optional three-rollout sentinel: 1
CANDIDATE_BATCH_SIZE = 8  # lower only after a GPU-memory error
OUTPUT_ROOT = Path('/content/drive/MyDrive/pnp_qplanning_corrector')
Q10_CHECKPOINT_PATH = None
Q50_CHECKPOINT_PATH = None

def resolve_checkpoint(horizon, explicit):
    if explicit is not None:
        path = Path(explicit)
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    matches = sorted(OUTPUT_ROOT.glob(
        f'pcpcds-*/q{horizon}_full/checkpoint_step_008000.pt'))
    if len(matches) != 1:
        raise ValueError(
            f'Expected exactly one Q{horizon} full checkpoint; found ' +
            f'{len(matches)}: {[str(path) for path in matches]}. ' +
            f'Set Q{horizon}_CHECKPOINT_PATH explicitly.')
    return matches[0]

Q10_CHECKPOINT_PATH = resolve_checkpoint(10, Q10_CHECKPOINT_PATH)
Q50_CHECKPOINT_PATH = resolve_checkpoint(50, Q50_CHECKPOINT_PATH)
print({'shard': f'{SHARD_INDEX}/{SHARD_COUNT}',
       'episode_limit': EPISODE_LIMIT,
       'candidate_batch_size': CANDIDATE_BATCH_SIZE,
       'q10_checkpoint': str(Q10_CHECKPOINT_PATH),
       'q50_checkpoint': str(Q50_CHECKPOINT_PATH)})

## 3. Run the three matched arms

In [ ]:
from pnp.qplanning_eval_experiment import run_qplanning_heldout_worker

report = run_qplanning_heldout_worker(
    q10_checkpoint_path=Q10_CHECKPOINT_PATH,
    q50_checkpoint_path=Q50_CHECKPOINT_PATH,
    shard_count=SHARD_COUNT, shard_index=SHARD_INDEX,
    episode_limit=EPISODE_LIMIT,
    candidate_batch_size=CANDIDATE_BATCH_SIZE)
report

## 4. Persisted three-arm audit

In [ ]:
from pnp.qplanning_eval_experiment import validate_qplanning_heldout_sentinel

validate_qplanning_heldout_sentinel(
    q10_checkpoint_id=report['q10_checkpoint_id'],
    q50_checkpoint_id=report['q50_checkpoint_id'],
    experiment=report['experiment'])